In [2]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np

print("All imports successful!")

All imports successful!


In [5]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:

from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HuggingFace")      # Change to HF_TOKEN if your secret has that name
login(token=token)

# ------------------------------------------------------------
# Import Libraries
# ------------------------------------------------------------

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import faiss
import numpy as np

# ------------------------------------------------------------
# Create Knowledge Base
# ------------------------------------------------------------

documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation (RAG) combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]

# ------------------------------------------------------------
# Generate Embeddings
# ------------------------------------------------------------

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embed_model.encode(documents)

# ------------------------------------------------------------
# Create FAISS Index
# ------------------------------------------------------------

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(doc_embeddings))

# ------------------------------------------------------------
# Retrieve Relevant Documents
# ------------------------------------------------------------

query = "What is RAG in AI?"

query_embedding = embed_model.encode([query])

D, I = index.search(np.array(query_embedding), k=2)

retrieved_chunks = [documents[i] for i in I[0]]

print("Retrieved Context:\n")

for chunk in retrieved_chunks:
    print("-", chunk)

# ------------------------------------------------------------
# Load FLAN-T5
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

# ------------------------------------------------------------
# Generate Answer
# ------------------------------------------------------------

context = " ".join(retrieved_chunks)

prompt = f"""
Use the context below to answer the question in one complete sentence.

Context:
{context}

Question:
{query}

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    temperature=0.3
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\nQuestion:", query)
print("\nAnswer:")
print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retrieved Context:

- Retrieval-Augmented Generation (RAG) combines document retrieval with text generation.
- Python is a popular high-level programming language used in AI development.


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Question: What is RAG in AI?

Answer:
combines document retrieval with text generation
